### Previsão de atraso na entrega e seu efeito na satisfação do cliente em e-commerce - Projeto Integrado em Ciência de Dados e Inteligência Artificial

In [1]:
import pandas as pd

In [2]:
orderItems = pd.read_csv("olist_order_items_dataset.csv")
orderPayments = pd.read_csv("olist_order_payments_dataset.csv")
orderReviews = pd.read_csv("olist_order_reviews_dataset.csv")
orderDataset = pd.read_csv("olist_orders_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
geolocation = pd.read_csv("olist_geolocation_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
nameTranslation = pd.read_csv("product_category_name_translation.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'olist_order_items_dataset.csv'

### Iniciando a análise exploratória dos dados (EDA)

In [ ]:
dfs = {
    "items": orderItems,
    "payments": orderPayments,
    "reviews": orderReviews,
    "orders": orderDataset,
    "customers": customers,
    "geolocation": geolocation,
    "products": products,
    "sellers": sellers,
    "nameTranslation": nameTranslation
}

for nome, df in dfs.items():
    print(nome, df.shape)

items (112650, 7)
payments (103886, 5)
reviews (99224, 7)
orders (99441, 8)
customers (99441, 5)
geolocation (1000163, 5)
products (32951, 9)
sellers (3095, 4)
nameTranslation (71, 2)


### Como o atraso se distribui?

In [ ]:
orderDataset.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### Aqui, temos as informações de:

* Status da entrega
* A data que o pedido foi realizado
* A data que o pedido foi aprovado
* A data que o vendedor despachou o pacote
* A data que o cliente recebeu de fato o produto
* A data estimada em que o pedido chegaria até o cliente

Naturalmente, se a data que o cliente recebeu for após a data estimada, isso se enquadra como um atraso. Daqui, conseguimos tirar uma label de "entregue atrasado" e "entregue no prazo".

As datas em que o pedido foi realizado, aprovado e despachado podem fazer parte de uma análise complementar.

Os pedidos taggados como "delivered" possuem as informações de data de postagem, data estimada de entrega e data em que a entrega aconteceu de fato. Mas, temos outras possibilidades: "invoiced", ou seja, que já foram pagos mas o vendedor ainda não enviou, e também os "shipped", que foram enviados mas ainda não chegaram.

In [ ]:
orderDataset["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Cara, quando eu olho pra isso aqui, caberia investigar mais o quanto podemos reduzir a proporcionalidade, ainda que não seja uma mudança significativa o suficiente. Mas acredito que algumas das entradas que estão como canceled, unavaiblable ou invoiced talvez não sejam tão úteis quanto se espera.

In [ ]:
orderDataset["atrasado"] = orderDataset["order_delivered_carrier_date"] > orderDataset["order_estimated_delivery_date"]

orderDataset["atrasado"].value_counts()

atrasado
False    98968
True       473
Name: count, dtype: int64